# Semaphore + Queue experiments

Goal: feel how backpressure works.

In [ ]:
import asyncio, time, random

async def call(i):
    await asyncio.sleep(random.uniform(0.05, 0.2))
    return i

async def with_cap(n, cap):
    sem = asyncio.Semaphore(cap)
    async def guarded(i):
        async with sem:
            return await call(i)
    t = time.perf_counter()
    await asyncio.gather(*(guarded(i) for i in range(n)))
    return (time.perf_counter()-t)*1000

for cap in (1, 2, 5, 10, 50):
    print(f'cap={cap:3}  ms={await with_cap(50, cap):.1f}')

In [ ]:
# Backpressure: full queue blocks producers
import asyncio, time
async def producer(q):
    for i in range(20):
        await q.put(i)
        print('produced', i)
    for _ in range(2):
        await q.put(None)

async def consumer(q, name):
    while True:
        x = await q.get()
        if x is None: return
        await asyncio.sleep(0.05)
        print(name, 'consumed', x)

q = asyncio.Queue(maxsize=3)  # tiny queue => producer must wait
await asyncio.gather(producer(q), consumer(q, 'A'), consumer(q, 'B'))